# Evaluate a trained CRN surrogate

Loads a checkpoint, generates novel random CRNs, and compares neural SDE rollouts against SSA ground truth.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Make the repo importable from the notebook
repo_root = Path().resolve().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from crn_surrogate.data.generation.mass_action_generator import MassActionCRNGenerator
from crn_surrogate.encoder.bipartite_gnn import BipartiteGNNEncoder
from crn_surrogate.encoder.tensor_repr import crn_to_tensor_repr
from crn_surrogate.simulation.gillespie import GillespieSSA
from crn_surrogate.simulation.trajectory import Trajectory
from crn_surrogate.simulator.neural_sde import CRNNeuralSDE
from crn_surrogate.simulator.sde_solver import EulerMaruyamaSolver
from experiments.configs.mass_action_3s import ExperimentConfig

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Load checkpoint

In [ ]:
CHECKPOINT_PATH = "path/to/checkpoint.pt"  # update this

cfg = ExperimentConfig()
model_config = cfg.build_model_config()

encoder = BipartiteGNNEncoder(model_config.encoder).to(DEVICE)
sde = CRNNeuralSDE(model_config.sde, n_species=cfg.max_n_species).to(DEVICE)

ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
encoder.load_state_dict(ckpt["encoder_state"])
sde.load_state_dict(ckpt["sde_state"])
encoder.eval()
sde.eval()
print("Checkpoint loaded.")

## Generate novel test CRNs

In [ ]:
torch.manual_seed(99)
gen = MassActionCRNGenerator(cfg.dataset.generator)
test_crns = gen.sample_batch(20)
print(f"Generated {len(test_crns)} test CRNs")
for i, crn in enumerate(test_crns[:5]):
    print(f"  CRN {i}: {crn.n_species} species, {crn.n_reactions} reactions")

## Compare SDE vs SSA for selected CRNs

In [ ]:
ssa = GillespieSSA()
solver = EulerMaruyamaSolver(model_config.sde)

T_MAX = cfg.dataset.t_max
N_TIME_POINTS = cfg.dataset.n_time_points
DT = 0.05
K_ROLLOUTS = 10
M_SSA = 20
times = torch.linspace(0.0, T_MAX, N_TIME_POINTS)

selected = test_crns[:6]  # plot 6 CRNs

fig_rows = len(selected)
fig_cols = max(crn.n_species for crn in selected)
fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(4 * fig_cols, 3 * fig_rows))
if fig_rows == 1:
    axes = axes[None, :]
if fig_cols == 1:
    axes = axes[:, None]

for row, crn in enumerate(selected):
    init_state = gen.sample_initial_state(crn).to(DEVICE)
    crn_repr = crn_to_tensor_repr(crn).to(DEVICE)

    # Encode with actual-sized state
    with torch.no_grad():
        ctx = encoder(crn_repr, init_state)

        # Pad initial state for SDE
        padded_init = torch.zeros(cfg.max_n_species, device=DEVICE)
        padded_init[: crn.n_species] = init_state

        # Neural SDE rollouts
        sde_trajs = []
        for _ in range(K_ROLLOUTS):
            traj = solver.solve(sde, padded_init, ctx, times.to(DEVICE), DT)
            sde_trajs.append(traj.states[:, : crn.n_species].cpu())
        sde_tensor = torch.stack(sde_trajs)  # (K, T, n_species)

    # SSA ground truth
    ssa_list = ssa.simulate_batch(
        stoichiometry=crn.stoichiometry_matrix,
        propensity_fn=crn.evaluate_propensities,
        initial_state=init_state.cpu(),
        t_max=T_MAX,
        n_trajectories=M_SSA,
    )
    ssa_tensor = Trajectory.stack_on_grid(ssa_list, times)  # (M, T, n_species)

    t_np = times.numpy()
    for s in range(crn.n_species):
        ax = axes[row, s]
        # SSA trajectories
        for m in range(M_SSA):
            ax.plot(
                t_np, ssa_tensor[m, :, s].numpy(), color="steelblue", alpha=0.15, lw=0.8
            )
        ax.plot(
            t_np,
            ssa_tensor[:, :, s].mean(0).numpy(),
            color="steelblue",
            lw=2,
            label="SSA mean",
        )
        # SDE rollouts
        for k in range(K_ROLLOUTS):
            ax.plot(
                t_np, sde_tensor[k, :, s].numpy(), color="tomato", alpha=0.25, lw=0.8
            )
        ax.plot(
            t_np,
            sde_tensor[:, :, s].mean(0).numpy(),
            color="tomato",
            lw=2,
            label="SDE mean",
        )
        ax.set_title(f"CRN {row} – {crn.species_names[s]}")
        ax.set_xlabel("time")
        ax.set_ylabel("count")
        if s == 0:
            ax.legend(fontsize=7)

    # Hide unused species columns
    for s in range(crn.n_species, fig_cols):
        axes[row, s].set_visible(False)

plt.tight_layout()
plt.savefig("sde_vs_ssa.pdf", dpi=150)
plt.show()

## Summary metrics table

In [ ]:
print(f"{'CRN':>5} {'n_sp':>4} {'n_rxn':>5} {'mean_mse':>10} {'var_mse':>10}")
print("-" * 40)

for i, crn in enumerate(selected):
    init_state = gen.sample_initial_state(crn).to(DEVICE)
    crn_repr = crn_to_tensor_repr(crn).to(DEVICE)

    with torch.no_grad():
        ctx = encoder(crn_repr, init_state)
        padded_init = torch.zeros(cfg.max_n_species, device=DEVICE)
        padded_init[: crn.n_species] = init_state
        sde_trajs = [
            solver.solve(sde, padded_init, ctx, times.to(DEVICE), DT)
            .states[:, : crn.n_species]
            .cpu()
            for _ in range(K_ROLLOUTS)
        ]
    sde_t = torch.stack(sde_trajs)

    ssa_list = ssa.simulate_batch(
        stoichiometry=crn.stoichiometry_matrix,
        propensity_fn=crn.evaluate_propensities,
        initial_state=init_state.cpu(),
        t_max=T_MAX,
        n_trajectories=M_SSA,
    )
    ssa_t = Trajectory.stack_on_grid(ssa_list, times)

    mean_mse = ((sde_t.mean(0) - ssa_t.mean(0)) ** 2).mean().item()
    var_mse = ((sde_t.var(0) - ssa_t.var(0)) ** 2).mean().item()
    print(
        f"{i:>5} {crn.n_species:>4} {crn.n_reactions:>5} {mean_mse:>10.4f} {var_mse:>10.4f}"
    )

## Topology diversity of test CRNs

In [ ]:
from collections import Counter

species_counts = Counter(crn.n_species for crn in test_crns)
reaction_counts = Counter(crn.n_reactions for crn in test_crns)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.bar(species_counts.keys(), species_counts.values())
ax1.set_xlabel("n_species")
ax1.set_ylabel("count")
ax1.set_title("Species distribution")

ax2.bar(reaction_counts.keys(), reaction_counts.values())
ax2.set_xlabel("n_reactions")
ax2.set_ylabel("count")
ax2.set_title("Reactions distribution")

plt.tight_layout()
plt.show()